Ran command: `python3 -c "
import json
with open('/Users/songchao/WorkSpace/Python/DeepLearn/pytorch/chapter_deep-learning-computation/model-construction.ipynb', 'r', encoding='utf-8') as f:
    nb = json.load(f)
for i, cell in enumerate(nb['cells']):
    cell_type = cell.get('cell_type')
    src = ''.join(cell.get('source', []))
    print(f'=== CELL {i} [{cell_type}] ===')
    print(src[:500])
"`
Ran command: `python3 -c "
import json
with open('/Users/songchao/WorkSpace/Python/DeepLearn/pytorch/chapter_deep-learning-computation/model-construction.ipynb', 'r', encoding='utf-8') as f:
    nb = json.load(f)
for i in range(7):
    cell = nb['cells'][i]
    cell_type = cell.get('cell_type')
    src = ''.join(cell.get('source', []))
    print(f'=== CELL {i} [{cell_type}] ===')
    print(src)
"`

本节内容来自 [model-construction.ipynb](file:///Users/songchao/WorkSpace/Python/DeepLearn/pytorch/chapter_deep-learning-computation/model-construction.ipynb)，主题为**层和块（Model Construction）**。它是深度学习计算的核心章节，深入剖析了 PyTorch 中神经网络组件的抽象方式（即 `nn.Module`）以及如何构建灵活、复杂的网络架构。

以下是对该 Notebook 的系统总结：

---

### 一、 核心概念：什么是“块”（Block）？

1. **抽象层次的演进**：
   * 单个神经元/单层网络 → 多层感知机（整个网络） → **块（Block）**。
   * 在现代复杂网络（如具有上百层的 ResNet-152）中，网络通常是由许多**重复出现的层组（groups of layers）**组成的。
   * “块”是一个更高层次的抽象：**它可以代表单个层、一组层，甚至整个模型本身**。利用块的递归嵌套，能够以极简的代码搭建极复杂的架构。

2. **一个块必须具备的 5 项基本功能**：
   * 接收输入数据作为前向传播的参数；
   * 通过前向传播计算生成输出（输出维度可与输入不同）；
   * 计算输出关于输入的梯度（反向传播，通常由 PyTorch Autograd 机制自动生成）；
   * 存储并访问前向计算所需的模型参数；
   * 根据需要初始化参数。

---

### 二、 核心内容与代码实践

#### 1. 自定义块：继承 `nn.Module`
通过继承 `nn.Module`，只需实现构造函数 `__init__` 和前向传播函数 `forward`：

```python
class MLP(nn.Module):
    def __init__(self):
        super().__init__()  # 必须调用父类构造函数进行初始化
        self.hidden = nn.Linear(20, 256)  # 隐藏层
        self.out = nn.Linear(256, 10)     # 输出层

    def forward(self, X):
        return self.out(F.relu(self.hidden(X)))
```
* **要点**：通过 `net(X)` 调用模型实际上执行了 `net.__call__(X)`，内部会触发 `forward` 方法；反向传播与参数管理由 PyTorch 自动处理。

---

#### 2. 深入理解顺序块：从零实现 `MySequential`
Notebook 解构了 `nn.Sequential` 的底层运作机制，手写了一个等价的简化版：

```python
class MySequential(nn.Module):
    def __init__(self, *args):
        super().__init__()
        for idx, module in enumerate(args):
            # 将子模块保存在 Module 父类的成员变量 _modules (OrderedDict) 中
            self._modules[str(idx)] = module

    def forward(self, X):
        for block in self._modules.values():
            X = block(X)
        return X
```

* **关键机制 —— 为什么用 `_modules` 而非普通 Python `list`？**
  * `self._modules` 是 `nn.Module` 内部用于追踪子模块的有序字典（`OrderedDict`）。
  * PyTorch 在调用 `net.parameters()` 收集参数、进行权重初始化、或执行 `.to(device)` / `.cuda()` 设备转移时，会自动递归遍历 `_modules` 中的子块。如果使用普通 Python 列表存储，PyTorch 将无法感知并注册这些子模块及其参数。

---

#### 3. 灵活性拓展：在前向传播中执行控制流与自定义计算
`nn.Sequential` 仅支持顺序级联，当网络需要动态分支、循环或非学习参数时，可在自定义块的 `forward` 中嵌入任意 Python 代码：

* **常数参数（Constant Parameters）**：既不是上层输入，也不是待优化的权重（设置 `requires_grad=False`），常用于固定投影或特定数学变换。
* **参数复用（Weight Sharing）**：多次调用同一个层实例（例如多次调用 `self.linear`），实现权重共享。
* **Python 控制流**：自由使用 `if-else`、`while` 循环进行条件控制。

```python
class FixedHiddenMLP(nn.Module):
    def __init__(self):
        super().__init__()
        # 常数参数：不参与梯度更新
        self.rand_weight = torch.rand((20, 20), requires_grad=False)
        self.linear = nn.Linear(20, 20)

    def forward(self, X):
        X = self.linear(X)
        X = F.relu(torch.mm(X, self.rand_weight) + 1)
        # 复用全连接层（参数共享）
        X = self.linear(X)
        # Python 控制流
        while X.abs().sum() > 1:
            X /= 2
        return X.sum()
```

---

#### 4. 混合搭配与嵌套块
块具有高度的可组合性，可以像乐高积木一样多层嵌套：
* 一个自定义块（如 `NestMLP`）内部可以包含 `nn.Sequential`；
* 另一个外层 `nn.Sequential` 又可以将自定义块、内置层与其他嵌套块串联成更大的复合网络（例如示例中的 `chimera` 嵌合体模型）。

---

### 三、 总结与思考

1. **核心设计思想**：
   * 模块化（Modularity）和层次化（Hierarchy）是构建复杂深度学习模型的核心。
   * `nn.Module` 作为基类，统一了“单层”、“模块”与“完整网络”的接口标准。
2. **效率考量**：
   * 在 PyTorch 中编写 Python 控制流虽然灵活，但过度依赖复杂的 CPU 动态控制可能会影响高速 GPU 计算的吞吐效率（后续章节通过 TorchScript / 静态图优化解决）。

In [1]:
import torch
from torch import nn
from torch.nn import functional as F

In [2]:
net = nn.Sequential(nn.Linear(20, 256), nn.ReLU(), nn.Linear(256, 10))
X = torch.rand(2, 20)
net(X)

tensor([[ 0.0835,  0.0822,  0.1875, -0.1669, -0.0624,  0.0617,  0.0977,  0.0183,
         -0.0610,  0.1950],
        [ 0.1468,  0.0501,  0.1889, -0.0824, -0.0119,  0.0433,  0.0745,  0.1110,
         -0.0537,  0.1516]], grad_fn=<AddmmBackward0>)

In [5]:
class MLP(nn.Module):

    def __init__(self):
        super().__init__()
        self.hidden = nn.Linear(20, 256)
        self.out = nn.Linear(256, 10)

    def forward(self, X):
        return self.out(F.relu(self.hidden(X)))


In [6]:
net = MLP()
net(X)

tensor([[ 0.0715,  0.1579, -0.1133, -0.1517,  0.1707, -0.0724, -0.1387, -0.1173,
          0.0604,  0.0764],
        [ 0.0461,  0.2596, -0.1241, -0.1693,  0.2207, -0.0096, -0.1015, -0.0576,
          0.0037,  0.0950]], grad_fn=<AddmmBackward0>)

In [7]:
class MySequential(nn.Module):

    def __init__(self, *args):
        super().__init__()
        for ind, module in enumerate(args):
            self._modules[str(ind)] = module
        
    def forward(self, X):
        for block in self._modules.values():
            X = block(X)
        return X

In [8]:
net = MySequential(nn.Linear(20, 256), nn.ReLU(), nn.Linear(256, 10))
net(X)

tensor([[-0.0865,  0.0244, -0.0546, -0.0510, -0.0862,  0.0469,  0.0928, -0.0407,
         -0.0618, -0.0329],
        [-0.0616, -0.2080, -0.0952, -0.1556,  0.0137,  0.0526,  0.0159,  0.0109,
         -0.0069, -0.0585]], grad_fn=<AddmmBackward0>)

In [9]:
class FixHiddenMLP(nn.Module):

    def __init__(self):
        super().__init__()
        self.rand_weight = torch.rand((20,20),  requires_grad=False)
        self.linear = nn.Linear(20, 20)

    def forward(self, X):
        X = self.linear(X)

        X = F.relu(torch.mm(X, self.rand_weight)+1)

        X = self.linear(X)

        while X.abs().sum() > 1:
            X /=2
        return X.sum()

In [10]:
net = FixHiddenMLP()
net(X)

tensor(0.3355, grad_fn=<SumBackward0>)

In [13]:
class NestMLP(nn.Module):

    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(20, 64), nn.ReLU(), nn.Linear(64, 32), nn.ReLU())
        self.linear = nn.Linear(32, 16)

    def forward(self, X):
        return self.linear(self.net(X))


In [15]:
chimera = nn.Sequential(NestMLP(), nn.Linear(16, 20), FixHiddenMLP())
chimera(X)

tensor(0.1808, grad_fn=<SumBackward0>)

以下是 [model-construction.ipynb](file:///Users/songchao/WorkSpace/Python/DeepLearn/pytorch/chapter_deep-learning-computation/model-construction.ipynb) 课后 3 道练习题的详细解析与代码实现。

---

### 练习 1：如果将 `MySequential` 中存储块的方式更改为 Python 列表，会出现什么样的问题？

#### 核心问题
如果使用普通的 Python `list`（如 `self.modules = list(args)`）存储子模块，**PyTorch 的 `nn.Module` 将无法感知并注册这些子模块及其参数**。

#### 具体引发的问题：
1. **参数无法被追踪 (`net.parameters()` 为空)**：
   * `nn.Module` 重写了 Python 的 `__setattr__` 方法。当我们将模块赋值给类属性（如 `self.linear = nn.Linear(...)`）或存入 `self._modules` 时，PyTorch 会自动完成参数注册。
   * 普通 Python `list` 是外部容器，PyTorch 不会递归检索其中的 `nn.Module`。因此调用 `net.parameters()` 返回空生成器，导致**优化器（如 `torch.optim.SGD`）无法获取任何待更新权重，模型根本无法训练**。
2. **设备转移（GPU/CPU）失效**：
   * 调用 `net.to('cuda')` 或 `net.cuda()` 时，因为找不到子模块，底层各层的权重依然留在 CPU 上，前向传播时会抛出设备不一致的运行时异常。
3. **训练/评估模式切换失效**：
   * 调用 `net.train()` 或 `net.eval()` 时，模式无法传播到子模块中的 `Dropout` 或 `BatchNorm` 层。
4. **模型权重无法保存与恢复**：
   * 调用 `net.state_dict()` 得到的是空字典，无法保存模型权重。

> **正解做法**：如果需要以列表形式管理子模块，应使用 PyTorch 提供的专门容器 **`nn.ModuleList`**（例如 `self.layers = nn.ModuleList(args)`），或者如教材中直接写入 `self._modules` 字典。

#### 代码验证对比：
```python
import torch
from torch import nn

# 错误写法：使用普通 Python list
class BadSequential(nn.Module):
    def __init__(self, *args):
        super().__init__()
        self.layers = list(args)  # 普通 Python 列表

    def forward(self, X):
        for layer in self.layers:
            X = layer(X)
        return X

bad_net = BadSequential(nn.Linear(20, 10))
print("BadSequential 参数量:", len(list(bad_net.parameters())))  # 输出: 0 (无法追踪！)
```

---

### 练习 2：实现一个块，它以两个块为参数（例如 `net1` 和 `net2`），并返回前向传播中两个网络的串联输出（平行块 Parallel Block）

#### 实现思路
* 构造函数接收两个子块，并将它们保存为类的属性以完成参数注册；
* 在 `forward` 中，将同一输入 `X` 分别传给 `net1` 和 `net2`；
* 使用 `torch.cat` 沿特征维度（`dim=1`）将两个分支的输出拼接（Concatenate）在一起。

#### 代码实现：
```python
import torch
from torch import nn

class ParallelBlock(nn.Module):
    def __init__(self, net1, net2, dim=1):
        super().__init__()
        # 赋值为属性，确保 PyTorch 自动将其注册为子模块并追踪参数
        self.net1 = net1
        self.net2 = net2
        self.dim = dim

    def forward(self, X):
        out1 = self.net1(X)
        out2 = self.net2(X)
        # 沿特征维度（dim=1）进行串联输出
        return torch.cat([out1, out2], dim=self.dim)

# --- 验证测试 ---
# 分支 1 输出维度为 16，分支 2 输出维度为 32
b1 = nn.Sequential(nn.Linear(20, 32), nn.ReLU(), nn.Linear(32, 16))
b2 = nn.Sequential(nn.Linear(20, 64), nn.ReLU(), nn.Linear(64, 32))

parallel_net = ParallelBlock(b1, b2)
X = torch.rand(4, 20)
output = parallel_net(X)

print("输入形状:", X.shape)            # torch.Size([4, 20])
print("并行输出形状:", output.shape)    # torch.Size([4, 48]) (16 + 32 = 48)
print("参数总数:", sum(p.numel() for p in parallel_net.parameters()))
```

---

### 练习 3：假设我们想要连接同一网络的多个实例。实现一个函数，该函数生成同一个块的多个实例，并在此基础上构建更大的网络。

#### 实现思路
* **避免浅引用陷阱**：在循环中不能直接 `[block] * n`，否则只是同一个实例的多次引用（会导致权重强制共享）。
* **独立实例的工厂函数**：可以通过传入**块的类对象（或工厂函数）及构造参数**，在循环体内独立调用构造函数，生成 `num_instances` 个相互独立、参数互不相干的子块，最后使用 `nn.Sequential` 串联。
* 为了能串联运行，重复连接的块通常需要保持**输入特征维度与输出特征维度一致**（即 `in_features == out_features`，这在 ResNet Block、Transformer Encoder 层中十分普遍）。

#### 代码实现：
```python
import torch
from torch import nn

def build_repeated_network(block_class, num_instances, *args, **kwargs):
    """
    生成同一个块的多个独立实例并使用 nn.Sequential 顺序串联。
    
    参数:
        block_class: 待实例化的块类 (如 ResidualBlock, MLP 等)
        num_instances: 重复实例的数量
        *args, **kwargs: 传递给 block_class 的构造参数
    """
    layers = []
    for _ in range(num_instances):
        # 每次循环都独立实例化一个全新的块对象，拥有各自独立的参数
        layers.append(block_class(*args, **kwargs))
    return nn.Sequential(*layers)


# --- 验证测试 ---
# 定义一个输入输出维度相同的残差/特征变换块
class FeatureBlock(nn.Module):
    def __init__(self, num_features):
        super().__init__()
        self.dense = nn.Linear(num_features, num_features)
        self.relu = nn.ReLU()

    def forward(self, X):
        return self.relu(self.dense(X)) + X  # 包含残差连接的块

# 构建一个由 5 个相同的 FeatureBlock 堆叠而成的大网络
num_layers = 5
features_dim = 20
deep_net = build_repeated_network(FeatureBlock, num_layers, num_features=features_dim)

print("大网络结构:\n", deep_net)

X = torch.rand(4, features_dim)
y = deep_net(X)
print("输入形状:", X.shape)
print("输出形状:", y.shape)  # torch.Size([4, 20])
```

#### 补充思考（参数独立 vs 参数共享）：
* **方案 A（上述代码，默认语义）**：各层拥有**完全独立**的权重参数，模型容量随深度线性增加。
* **方案 B（循环权重共享 / Weight Sharing）**：如果某种算法专门要求参数共享（例如循环神经网络 RNN 的单步逻辑），则只需创建**一个**实例并在 `forward` 循环中多次调用该实例。